In [ ]:

from google.colab import drive
from pathlib import Path

# Ket noi Google Drive 5 TB
drive.mount("/content/drive")

# Thu muc luu du lieu
DRIVE_DIR = Path(
    "/content/drive/MyDrive/"
    "onchain-whale-analysis/data/market/trades"
)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Da ket noi Google Drive!")
print("Thu muc:", DRIVE_DIR)


In [ ]:

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zipfile
import time
import json
import gc
from pathlib import Path

# ===== CAU HINH T06 =====
SYMBOL = "ETHUSDT"
START_DATE = "2025-01-02"
END_DATE = "2025-01-04"  # Thu 1 ngay truoc
CHUNK_SIZE = 100_000

BASE_URL = "https://data.binance.vision/data/spot/daily/trades"

# Google Drive: noi luu chinh
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market"
)
OUTPUT_DIR = DRIVE_ROOT / "trades"
LOG_DIR = DRIVE_ROOT / "logs"

# Bo nho tam tren may chu Colab, KHONG PHAI LAPTOP
TEMP_DIR = Path("/content/t06_temp")

for folder in [OUTPUT_DIR, LOG_DIR, TEMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MANIFEST = LOG_DIR / "t06_download_manifest.jsonl"

COLUMNS = [
    "trade_id", "price", "quantity",
    "quote_quantity", "trade_time",
    "is_buyer_maker", "is_best_match"
]


def save_log(record):
    with open(MANIFEST, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def download_day(day):
    date_str = day.strftime("%Y-%m-%d")

    folder = (
        OUTPUT_DIR
        / f"year={day.year}"
        / f"month={day.month:02d}"
    )
    folder.mkdir(parents=True, exist_ok=True)

    output = folder / f"{SYMBOL}_trades_{date_str}.parquet"
    temporary_output = folder / f"{SYMBOL}_trades_{date_str}.part"
    temporary_zip = TEMP_DIR / f"{SYMBOL}_{date_str}.zip"

    # Chi bo qua file Parquet da ghi hoan chinh
    if output.exists():
        try:
            meta = pq.read_metadata(output)
            if meta.num_rows > 0:
                print(f"SKIP {date_str}: da co {meta.num_rows:,} trades")
                return "skipped"
        except Exception:
            print(f"File {date_str} bi loi, se tai lai")

    url = (
        f"{BASE_URL}/{SYMBOL}/"
        f"{SYMBOL}-trades-{date_str}.zip"
    )

    writer = None
    total_rows = 0
    min_time = None
    max_time = None

    try:
        # 1. Tai ZIP vao bo nho tam Colab
        with requests.get(
            url, stream=True, timeout=(20, 120)
        ) as response:
            response.raise_for_status()
            with open(temporary_zip, "wb") as f:
                for block in response.iter_content(1024 * 1024):
                    if block:
                        f.write(block)

        print(
            f"Downloaded {date_str}: "
            f"{temporary_zip.stat().st_size / 1024**2:.2f} MB ZIP"
        )

        # 2. Doc tung phan, ghi Parquet vao Drive
        with zipfile.ZipFile(temporary_zip) as z:
            csv_name = next(
                x for x in z.namelist()
                if x.endswith(".csv")
            )

            with z.open(csv_name) as f:
                first_line = f.readline().decode(
                    "utf-8-sig"
                ).strip().lower()

            has_header = first_line.startswith(
                ("id,", "trade_id,")
            )

            with z.open(csv_name) as f:
                chunks = pd.read_csv(
                    f,
                    header=0 if has_header else None,
                    names=COLUMNS,
                    chunksize=CHUNK_SIZE
                )

                for chunk in chunks:
                    chunk["trade_id"] = pd.to_numeric(
                        chunk["trade_id"], errors="raise"
                    ).astype("int64")

                    chunk["trade_time"] = pd.to_numeric(
                        chunk["trade_time"], errors="raise"
                    ).astype("int64")

                    for col in ["price", "quantity", "quote_quantity"]:
                        chunk[col] = pd.to_numeric(
                            chunk[col], errors="raise"
                        )

                    chunk["datetime_utc"] = pd.to_datetime(
                        chunk["trade_time"], unit="us", utc=True
                    )
                    chunk["symbol"] = SYMBOL

                    cmin = chunk["datetime_utc"].min()
                    cmax = chunk["datetime_utc"].max()
                    min_time = cmin if min_time is None else min(min_time, cmin)
                    max_time = cmax if max_time is None else max(max_time, cmax)

                    table = pa.Table.from_pandas(
                        chunk, preserve_index=False
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            str(temporary_output),
                            table.schema,
                            compression="zstd"
                        )

                    writer.write_table(table)
                    total_rows += len(chunk)

        if writer:
            writer.close()
            writer = None

        # 3. Kiem tra file
        if total_rows == 0:
            raise ValueError("File khong co du lieu")

        if (
            min_time.strftime("%Y-%m-%d") != date_str
            or max_time.strftime("%Y-%m-%d") != date_str
        ):
            raise ValueError("Timestamp nam ngoai ngay")

        metadata = pq.read_metadata(temporary_output)
        if metadata.num_rows != total_rows:
            raise ValueError("So dong Parquet khong khop")

        # 4. Doi ten file sau khi ghi hoan chinh
        temporary_output.replace(output)

        save_log({
            "date": date_str,
            "status": "success",
            "rows": total_rows,
            "source": url,
            "parquet_mb": round(
                output.stat().st_size / 1024**2, 3
            )
        })

        print(
            f"OK {date_str} | {total_rows:,} trades | "
            f"{output.stat().st_size / 1024**2:.2f} MB"
        )
        return "success"

    except Exception as e:
        if writer:
            writer.close()

        print(f"ERROR {date_str}: {e}")
        save_log({
            "date": date_str,
            "status": "failed",
            "error": str(e)
        })
        return "failed"

    finally:
        temporary_zip.unlink(missing_ok=True)
        temporary_output.unlink(missing_ok=True)
        gc.collect()


# ===== BAT DAU CHAY =====
days = pd.date_range(START_DATE, END_DATE, freq="D")

summary = {"success": 0, "skipped": 0, "failed": 0}

for day in days:
    status = download_day(day)
    summary[status] += 1
    time.sleep(0.5)

print("\n===== T06 SUMMARY =====")
print(summary)
print("Google Drive:", OUTPUT_DIR)


In [ ]:
# ===== T06: TAI DU LIEU THEO TUNG DOT =====

import time
import pandas as pd
from pathlib import Path

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

# Moi dot xu ly toi da 15 ngay chua co du lieu
BATCH_SIZE = 15

# Kiem tra file da tai
def is_completed(day):
    date_str = day.strftime("%Y-%m-%d")

    path = (
        OUTPUT_DIR
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"{SYMBOL}_trades_{date_str}.parquet"
    )

    if not path.exists():
        return False

    try:
        metadata = pq.read_metadata(path)
        return metadata.num_rows > 0
    except Exception:
        return False


all_days = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

pending_days = [
    day for day in all_days
    if not is_completed(day)
]

print("Tong ngay nghien cuu:", len(all_days))
print("Ngay da co du lieu:", len(all_days) - len(pending_days))
print("Ngay chua tai:", len(pending_days))

batch = pending_days[:BATCH_SIZE]

summary = {
    "success": 0,
    "skipped": 0,
    "failed": 0
}

print("\nBAT DAU TAI", len(batch), "NGAY")

for i, day in enumerate(batch, 1):
    print(
        f"\n[{i}/{len(batch)}] "
        f"{day.strftime('%Y-%m-%d')}"
    )

    status = download_day(day)
    summary[status] += 1

    time.sleep(0.5)

print("\n===== T06 BATCH SUMMARY =====")
print(summary)
print("Con lai:", len(pending_days) - summary["success"])
print("Du lieu luu tai:", OUTPUT_DIR)


In [ ]:

# ===== T06: TAI DU LIEU THEO TUNG DOT =====

import time
import pandas as pd
from pathlib import Path

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

# Moi dot xu ly toi da 15 ngay chua co du lieu
BATCH_SIZE = 15

# Kiem tra file da tai
def is_completed(day):
    date_str = day.strftime("%Y-%m-%d")

    path = (
        OUTPUT_DIR
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"{SYMBOL}_trades_{date_str}.parquet"
    )

    if not path.exists():
        return False

    try:
        metadata = pq.read_metadata(path)
        return metadata.num_rows > 0
    except Exception:
        return False


all_days = pd.date_range(
    START_DATE,
    END_DATE,
    freq="D"
)

pending_days = [
    day for day in all_days
    if not is_completed(day)
]

print("Tong ngay nghien cuu:", len(all_days))
print("Ngay da co du lieu:", len(all_days) - len(pending_days))
print("Ngay chua tai:", len(pending_days))

batch = pending_days[:BATCH_SIZE]

summary = {
    "success": 0,
    "skipped": 0,
    "failed": 0
}

print("\nBAT DAU TAI", len(batch), "NGAY")

for i, day in enumerate(batch, 1):
    print(
        f"\n[{i}/{len(batch)}] "
        f"{day.strftime('%Y-%m-%d')}"
    )

    status = download_day(day)
    summary[status] += 1

    time.sleep(0.5)

print("\n===== T06 BATCH SUMMARY =====")
print(summary)
print("Con lai:", len(pending_days) - summary["success"])
print("Du lieu luu tai:", OUTPUT_DIR)


In [ ]:

import time
import pandas as pd
import pyarrow.parquet as pq

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

# Toi da 60 ngay moi lan chay
MAX_DAYS = 60

# Dung sau 90 phut de tranh phien chay qua dai
MAX_RUNTIME = 90 * 60

def is_completed(day):
    date_str = day.strftime("%Y-%m-%d")

    path = (
        OUTPUT_DIR
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"{SYMBOL}_trades_{date_str}.parquet"
    )

    if not path.exists():
        return False

    try:
        return pq.read_metadata(path).num_rows > 0
    except Exception:
        return False

days = pd.date_range(
    START_DATE, END_DATE, freq="D"
)

pending = [
    day for day in days
    if not is_completed(day)
]

print("Tong ngay:", len(days))
print("Da hoan thanh:", len(days) - len(pending))
print("Con lai:", len(pending))

start_time = time.monotonic()
summary = {"success": 0, "skipped": 0, "failed": 0}

for i, day in enumerate(pending[:MAX_DAYS], 1):

    if time.monotonic() - start_time > MAX_RUNTIME:
        print("Dung do dat gioi han thoi gian.")
        break

    print(
        f"\n[{i}/{min(MAX_DAYS, len(pending))}] "
        f"{day.strftime('%Y-%m-%d')}"
    )

    status = download_day(day)
    summary[status] += 1

    time.sleep(1)

print("\n===== T06 SUMMARY =====")
print(summary)
print("Da luu du lieu vao Google Drive.")


In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market/trades"
)

days = pd.date_range("2025-01-01", "2025-03-20")

missing = []
invalid = []
valid = []
total_trades = 0

for day in days:
    date_str = day.strftime("%Y-%m-%d")
    file = (
        ROOT
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"ETHUSDT_trades_{date_str}.parquet"
    )

    if not file.exists():
        missing.append(date_str)
        continue

    try:
        metadata = pq.read_metadata(file)
        if metadata.num_rows <= 0:
            invalid.append(date_str)
        else:
            valid.append(date_str)
            total_trades += metadata.num_rows
    except Exception:
        invalid.append(date_str)

print("===== T06 DATA QUALITY =====")
print("Tong so ngay:", len(days))
print("Ngay hop le (metadata):", len(valid))
print("Ngay thieu:", len(missing))
print("Ngay file loi:", len(invalid))
print("Tong giao dich:", f"{total_trades:,}")
print("Danh sach ngay thieu:", missing)
print("Danh sach file loi:", invalid)



In [ ]:

import time
import pandas as pd
import pyarrow.parquet as pq

# ===== T06: TIEP TUC TAI DU LIEU =====

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

MAX_DAYS = 60
MAX_RUNTIME = 90 * 60  # 90 phut

def is_completed(day):
    date_str = day.strftime("%Y-%m-%d")

    path = (
        OUTPUT_DIR
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"{SYMBOL}_trades_{date_str}.parquet"
    )

    if not path.exists():
        return False

    try:
        return pq.read_metadata(path).num_rows > 0
    except Exception:
        return False

all_days = pd.date_range(START_DATE, END_DATE, freq="D")

pending = [day for day in all_days if not is_completed(day)]

print("===== T06 PROGRESS =====")
print("Tong so ngay:", len(all_days))
print("Da co du lieu:", len(all_days) - len(pending))
print("Con lai:", len(pending))

batch = pending[:MAX_DAYS]
start_time = time.monotonic()

summary = {"success": 0, "skipped": 0, "failed": 0}

for i, day in enumerate(batch, 1):
    if time.monotonic() - start_time >= MAX_RUNTIME:
        print("Da dat gioi han thoi gian, dung an toan.")
        break

    print(f"\n[{i}/{len(batch)}] Dang xu ly {day:%Y-%m-%d}")

    status = download_day(day)
    summary[status] += 1

    time.sleep(1)

# Kiem tra lai tien do
remaining = sum(not is_completed(day) for day in all_days)

print("\n===== T06 FINAL SUMMARY =====")
print(summary)
print("Tong ngay da co file:", len(all_days) - remaining)
print("So ngay con lai:", remaining)
print("Du lieu tren Google Drive:", OUTPUT_DIR)


In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market/trades"
)

days = pd.date_range("2025-01-01", "2026-10-17")

valid = []
missing = []
invalid = []
total_trades = 0
total_bytes = 0

for day in days:
    date_str = day.strftime("%Y-%m-%d")

    path = (
        ROOT
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"ETHUSDT_trades_{date_str}.parquet"
    )

    if not path.exists():
        missing.append(date_str)
        continue

    try:
        meta = pq.read_metadata(path)

        if meta.num_rows <= 0:
            invalid.append(date_str)
            continue

        valid.append(date_str)
        total_trades += meta.num_rows
        total_bytes += path.stat().st_size

    except Exception:
        invalid.append(date_str)

print("========== T06 FINAL DATA CHECK ==========")
print("So ngay yeu cau:", len(days))
print("So ngay co du lieu:", len(valid))
print("So ngay thieu:", len(missing))
print("So file loi:", len(invalid))
print("Tong giao dich:", f"{total_trades:,}")
print("Dung luong Parquet (GiB):", round(total_bytes / 1024**3, 2))
print("Ngay thieu:", missing)
print("File loi:", invalid)

In [ ]:

from pathlib import Path
import json
from datetime import datetime, timezone

report_path = Path(
    "/content/drive/MyDrive/"
    "onchain-whale-analysis/data/market/logs/"
    "t06_collection_summary.json"
)

report_path.parent.mkdir(parents=True, exist_ok=True)

summary = {
    "task": "T06",
    "symbol": "ETHUSDT",
    "market": "Binance Spot",
    "source": "Binance Vision",
    "start_date": "2025-01-01",
    "end_date": "2026-10-17",
    "timezone": "UTC",
    "expected_days": 655,
    "available_days": 655,
    "missing_days": 0,
    "invalid_files_metadata": 0,
    "total_trades": 2435744778,
    "parquet_size_gib": 24.33,
    "validation_level": "metadata_only",
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

report_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Da luu bao cao:", report_path)


In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import time

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market/trades"
)

results = []

days = pd.date_range("2025-01-01", "2025-01-05")

for day in days:
    date_str = day.strftime("%Y-%m-%d")
    path = (
        ROOT
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"ETHUSDT_trades_{date_str}.parquet"
    )

    start = time.time()
    parquet = pq.ParquetFile(path)

    rows = 0
    null_count = 0
    invalid_price = 0
    invalid_quantity = 0
    wrong_date = 0
    duplicate_ids = 0
    previous_id = None

    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=[
            "trade_id", "price", "quantity",
            "datetime_utc"
        ]
    ):
        df = batch.to_pandas()
        rows += len(df)

        null_count += df.isna().sum().sum()
        invalid_price += (df["price"] <= 0).sum()
        invalid_quantity += (df["quantity"] <= 0).sum()

        wrong_date += (
            df["datetime_utc"].dt.strftime("%Y-%m-%d")
            != date_str
        ).sum()

        ids = df["trade_id"]

        duplicate_ids += ids.duplicated().sum()

        if previous_id is not None:
            duplicate_ids += int(ids.iloc[0] == previous_id)

        previous_id = ids.iloc[-1]

    results.append({
        "date": date_str,
        "rows": rows,
        "null_values": int(null_count),
        "invalid_price": int(invalid_price),
        "invalid_quantity": int(invalid_quantity),
        "wrong_date": int(wrong_date),
        "duplicate_ids_detected": int(duplicate_ids),
        "seconds": round(time.time() - start, 2)
    })

df_quality = pd.DataFrame(results)
display(df_quality)

print("\n===== DQC SUMMARY =====")
print("Checked days:", len(df_quality))
print("Checked trades:", f"{df_quality['rows'].sum():,}")
print("Null values:", df_quality["null_values"].sum())
print("Invalid prices:", df_quality["invalid_price"].sum())
print("Invalid quantities:", df_quality["invalid_quantity"].sum())
print("Wrong dates:", df_quality["wrong_date"].sum())
print("Detected duplicate IDs:", df_quality["duplicate_ids_detected"].sum())


In [ ]:

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import time

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market/trades"
)

REPORT_DIR = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market/processed"
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_FILE = REPORT_DIR / "t06_data_quality.csv"

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

MAX_DAYS = 30
BATCH_SIZE = 100_000

REQUIRED = [
    "trade_id", "price", "quantity",
    "quote_quantity", "trade_time",
    "is_buyer_maker", "is_best_match",
    "datetime_utc", "symbol"
]

# Tim cac ngay da kiem tra
if REPORT_FILE.exists():
    old = pd.read_csv(REPORT_FILE)
    completed = set(
        old.loc[old["status"] == "success", "date"]
        .astype(str)
    )
else:
    completed = set()

all_days = pd.date_range(START_DATE, END_DATE)

pending = [
    day for day in all_days
    if day.strftime("%Y-%m-%d") not in completed
]

print("Tong ngay:", len(all_days))
print("Da kiem tra:", len(all_days) - len(pending))
print("Con lai:", len(pending))

for day in pending[:MAX_DAYS]:
    date_str = day.strftime("%Y-%m-%d")
    start = time.time()

    path = (
        ROOT
        / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"ETHUSDT_trades_{date_str}.parquet"
    )

    result = {
        "date": date_str,
        "status": "success",
        "rows": 0,
        "null_values": 0,
        "invalid_price": 0,
        "invalid_quantity": 0,
        "wrong_date": 0,
        "duplicate_or_unordered_ids": 0,
        "id_gaps": 0,
        "time_reversals": 0,
        "first_id": None,
        "last_id": None,
        "seconds": 0,
        "error": ""
    }

    try:
        pf = pq.ParquetFile(path)

        missing_cols = set(REQUIRED) - set(pf.schema_arrow.names)
        if missing_cols:
            raise ValueError(f"Missing columns: {missing_cols}")

        previous_id = None
        previous_time = None

        for batch in pf.iter_batches(
            batch_size=BATCH_SIZE,
            columns=REQUIRED
        ):
            df = batch.to_pandas()
            result["rows"] += len(df)
            result["null_values"] += int(
                df.isna().sum().sum()
            )

            result["invalid_price"] += int(
                (df["price"] <= 0).sum()
            )
            result["invalid_quantity"] += int(
                (df["quantity"] <= 0).sum()
            )

            dates = pd.to_datetime(
                df["datetime_utc"], utc=True
            ).dt.strftime("%Y-%m-%d")

            result["wrong_date"] += int(
                (dates != date_str).sum()
            )

            ids = df["trade_id"].to_numpy(
                dtype=np.int64
            )
            timestamps = df["trade_time"].to_numpy(
                dtype=np.int64
            )

            if result["first_id"] is None and len(ids):
                result["first_id"] = int(ids[0])

            if previous_id is not None:
                ids = np.insert(ids, 0, previous_id)

            differences = np.diff(ids)

            result["duplicate_or_unordered_ids"] += int(
                (differences <= 0).sum()
            )
            result["id_gaps"] += int(
                (differences > 1).sum()
            )

            if previous_time is not None:
                timestamps = np.insert(
                    timestamps, 0, previous_time
                )

            result["time_reversals"] += int(
                (np.diff(timestamps) < 0).sum()
            )

            previous_id = int(ids[-1])
            previous_time = int(timestamps[-1])

        result["last_id"] = previous_id
        result["seconds"] = round(
            time.time() - start, 2
        )

        if result["rows"] != pf.metadata.num_rows:
            raise ValueError("Row count mismatch")

        # Luu ket qua sau moi ngay
        pd.DataFrame([result]).to_csv(
            REPORT_FILE,
            mode="a",
            header=not REPORT_FILE.exists(),
            index=False
        )

        print(
            f"OK {date_str} | "
            f"{result['rows']:,} trades | "
            f"{result['seconds']}s"
        )

    except Exception as e:
        print(f"ERROR {date_str}: {e}")

print("\n===== DQC PROGRESS =====")

if REPORT_FILE.exists():
    report = pd.read_csv(REPORT_FILE)
    print("Ngay kiem tra:", len(report))
    print("Tong giao dich:", f"{report['rows'].sum():,}")
    for col in [
        "null_values",
        "invalid_price",
        "invalid_quantity",
        "wrong_date",
        "duplicate_or_unordered_ids",
        "id_gaps",
        "time_reversals"
    ]:
        print(col + ":", report[col].sum())

print("Bao cao:", REPORT_FILE)

In [ ]:

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import time

BASE = Path("/content/drive/MyDrive/onchain-whale-analysis/data/market")
RAW = BASE / "trades"
MINUTE_DIR = BASE / "processed" / "ohlcv_1m"
DAILY_DIR = BASE / "processed" / "ohlcv_1d"

MINUTE_DIR.mkdir(parents=True, exist_ok=True)
DAILY_DIR.mkdir(parents=True, exist_ok=True)

def aggregate_day(date_str):
    day = pd.Timestamp(date_str)

    source = (
        RAW / f"year={day.year}"
        / f"month={day.month:02d}"
        / f"ETHUSDT_trades_{date_str}.parquet"
    )

    minute_file = MINUTE_DIR / f"ETHUSDT_1m_{date_str}.parquet"
    daily_file = DAILY_DIR / f"ETHUSDT_1d_{date_str}.parquet"

    # Khong xu ly lai ngay da co du hai file
    if minute_file.exists() and daily_file.exists():
        print(f"SKIP {date_str}")
        return

    start = time.time()
    parquet = pq.ParquetFile(source)
    partials = []

    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=[
            "price", "quantity", "quote_quantity",
            "datetime_utc", "is_buyer_maker"
        ]
    ):
        df = batch.to_pandas()

        df["minute"] = pd.to_datetime(
            df["datetime_utc"], utc=True
        ).dt.floor("min")

        # is_buyer_maker=True:
        # buyer la maker, ben ban la taker
        df["buy_eth"] = np.where(
            ~df["is_buyer_maker"].astype(bool),
            df["quantity"], 0.0
        )
        df["sell_eth"] = np.where(
            df["is_buyer_maker"].astype(bool),
            df["quantity"], 0.0
        )

        df["buy_usdt"] = np.where(
            ~df["is_buyer_maker"].astype(bool),
            df["quote_quantity"], 0.0
        )
        df["sell_usdt"] = np.where(
            df["is_buyer_maker"].astype(bool),
            df["quote_quantity"], 0.0
        )

        grouped = df.groupby(
            "minute", sort=True
        ).agg(
            open=("price", "first"),
            high=("price", "max"),
            low=("price", "min"),
            close=("price", "last"),
            volume_eth=("quantity", "sum"),
            volume_usdt=("quote_quantity", "sum"),
            trade_count=("price", "size"),
            buy_volume_eth=("buy_eth", "sum"),
            sell_volume_eth=("sell_eth", "sum"),
            buy_volume_usdt=("buy_usdt", "sum"),
            sell_volume_usdt=("sell_usdt", "sum")
        ).reset_index()

        partials.append(grouped)

    # Gop cac phut nam tren ranh gioi batch
    combined = pd.concat(partials, ignore_index=True)

    minute = combined.groupby(
        "minute", sort=False
    ).agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume_eth=("volume_eth", "sum"),
        volume_usdt=("volume_usdt", "sum"),
        trade_count=("trade_count", "sum"),
        buy_volume_eth=("buy_volume_eth", "sum"),
        sell_volume_eth=("sell_volume_eth", "sum"),
        buy_volume_usdt=("buy_volume_usdt", "sum"),
        sell_volume_usdt=("sell_volume_usdt", "sum")
    ).sort_index().reset_index()

    minute["imbalance_eth"] = (
        (minute["buy_volume_eth"] - minute["sell_volume_eth"])
        / minute["volume_eth"].replace(0, np.nan)
    )

    minute["vwap"] = (
        minute["volume_usdt"]
        / minute["volume_eth"].replace(0, np.nan)
    )

    # Tong hop du lieu 1 ngay tu du lieu 1 phut
    daily = pd.DataFrame([{
        "date": date_str,
        "open": minute["open"].iloc[0],
        "high": minute["high"].max(),
        "low": minute["low"].min(),
        "close": minute["close"].iloc[-1],
        "volume_eth": minute["volume_eth"].sum(),
        "volume_usdt": minute["volume_usdt"].sum(),
        "trade_count": int(minute["trade_count"].sum()),
        "buy_volume_eth": minute["buy_volume_eth"].sum(),
        "sell_volume_eth": minute["sell_volume_eth"].sum(),
        "buy_volume_usdt": minute["buy_volume_usdt"].sum(),
        "sell_volume_usdt": minute["sell_volume_usdt"].sum()
    }])

    daily["imbalance_eth"] = (
        (daily["buy_volume_eth"] - daily["sell_volume_eth"])
        / daily["volume_eth"]
    )
    daily["vwap"] = daily["volume_usdt"] / daily["volume_eth"]

    # Ghi file tam truoc, sau do moi doi ten
    minute_tmp = minute_file.with_suffix(".part")
    daily_tmp = daily_file.with_suffix(".part")

    minute.to_parquet(minute_tmp, index=False)
    daily.to_parquet(daily_tmp, index=False)

    # Xac nhan so giao dich khop voi file goc
    if int(minute["trade_count"].sum()) != parquet.metadata.num_rows:
        raise ValueError(f"Row count mismatch: {date_str}")

    minute_tmp.replace(minute_file)
    daily_tmp.replace(daily_file)

    print(
        f"OK {date_str} | "
        f"{len(minute)} minutes | "
        f"{int(daily['trade_count'].iloc[0]):,} trades | "
        f"{time.time() - start:.1f}s"
    )

# THU 3 NGAY TRUOC
for d in pd.date_range("2025-01-01", "2025-01-03"):
    aggregate_day(d.strftime("%Y-%m-%d"))

print("\nHOAN THANH MARKET AGGREGATION TEST")


In [ ]:

import pandas as pd
import time
from pathlib import Path

START_DATE = "2025-01-01"
END_DATE = "2026-10-17"

# Xu ly toi da 100 ngay moi lan chay
MAX_DAYS = 100

all_days = pd.date_range(START_DATE, END_DATE)

def is_aggregated(day):
    date_str = day.strftime("%Y-%m-%d")

    minute_file = (
        MINUTE_DIR / f"ETHUSDT_1m_{date_str}.parquet"
    )
    daily_file = (
        DAILY_DIR / f"ETHUSDT_1d_{date_str}.parquet"
    )

    # Chi bo qua neu ca hai file doc duoc metadata
    try:
        return (
            pq.read_metadata(minute_file).num_rows > 0
            and pq.read_metadata(daily_file).num_rows == 1
        )
    except Exception:
        return False

pending = [
    day for day in all_days
    if not is_aggregated(day)
]

print("===== T06 MARKET AGGREGATION =====")
print("Tong ngay:", len(all_days))
print("Da xu ly:", len(all_days) - len(pending))
print("Con lai:", len(pending))

summary = {"success": 0, "failed": 0}

for i, day in enumerate(pending[:MAX_DAYS], 1):
    date_str = day.strftime("%Y-%m-%d")

    try:
        aggregate_day(date_str)

        if not is_aggregated(day):
            raise ValueError("Output validation failed")

        summary["success"] += 1

    except Exception as e:
        summary["failed"] += 1
        print(f"ERROR {date_str}: {e}")

    if i % 10 == 0:
        print(f"Progress: {i}/{min(MAX_DAYS, len(pending))}")

remaining = sum(
    not is_aggregated(day) for day in all_days
)

print("\n===== AGGREGATION SUMMARY =====")
print(summary)
print("Tong ngay hoan thanh:", len(all_days) - remaining)
print("Con lai:", remaining)
print("Output 1m:", MINUTE_DIR)
print("Output 1d:", DAILY_DIR)


In [ ]:

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/"
    "data/market/processed/ohlcv_1m"
)

files = sorted(ROOT.glob("ETHUSDT_1m_*.parquet"))

total_rows = 0
bad_files = []

for file in files:
    try:
        meta = pq.read_metadata(file)
        total_rows += meta.num_rows
        if meta.num_rows != 1440:
            bad_files.append((file.name, meta.num_rows))
    except Exception as e:
        bad_files.append((file.name, str(e)))

print("===== T06 OHLCV CHECK =====")
print("So file:", len(files))
print("Tong dong 1m:", f"{total_rows:,}")
print("File khong du 1440 dong:", len(bad_files))
print("Chi tiet:", bad_files[:10])


In [ ]:

from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/"
    "data/market/processed/ohlcv_1m"
)

files = sorted(folder.glob("ETHUSDT_1m_*.parquet"))

sample = pd.read_parquet(files[0])

print("===== THONG TIN FILE OHLCV 1M =====")
print("File:", files[0].name)
print("\nDanh sach cot:")
print(sample.columns.tolist())

print("\n5 dong dau:")
display(sample.head())

print("\nKieu du lieu:")
print(sample.dtypes)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# ===== T06: FINAL MARKET AGGREGATION =====

BASE = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/"
    "data/market/processed"
)

INPUT_DIR = BASE / "ohlcv_1m"
HOUR_DIR = BASE / "ohlcv_1h"
DAY_DIR = BASE / "ohlcv_1d"

HOUR_DIR.mkdir(parents=True, exist_ok=True)
DAY_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(INPUT_DIR.glob("ETHUSDT_1m_*.parquet"))

print("So file 1m:", len(files))

if len(files) != 655:
    raise ValueError("Chua du 655 file OHLCV 1m")

# 1. Doc du lieu 1 phut
df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

df["minute"] = pd.to_datetime(df["minute"], utc=True)
df = df.sort_values("minute").set_index("minute")

print("So dong 1m:", f"{len(df):,}")

# 2. Quy tac tong hop
AGG = {
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume_eth": "sum",
    "volume_usdt": "sum",
    "trade_count": "sum",
    "buy_volume_eth": "sum",
    "sell_volume_eth": "sum",
    "buy_volume_usdt": "sum",
    "sell_volume_usdt": "sum"
}

def make_ohlcv(freq):
    result = df.resample(
        freq, label="left", closed="left"
    ).agg(AGG)

    result = result.dropna(
        subset=["open", "close"]
    )

    # VWAP = Quote Volume / Base Volume
    result["vwap"] = np.where(
        result["volume_eth"] > 0,
        result["volume_usdt"] / result["volume_eth"],
        np.nan
    )

    # Volume Imbalance
    result["imbalance_eth"] = np.where(
        result["volume_eth"] > 0,
        (
            result["buy_volume_eth"]
            - result["sell_volume_eth"]
        ) / result["volume_eth"],
        np.nan
    )

    result = result.reset_index().rename(
        columns={"minute": "datetime_utc"}
    )

    result["symbol"] = "ETHUSDT"

    return result

# 3. OHLCV 1 gio
hourly = make_ohlcv("1h")

hour_file = (
    HOUR_DIR / "ETHUSDT_1h_2025-2026.parquet"
)

hourly.to_parquet(
    hour_file,
    index=False,
    compression="zstd"
)

# 4. OHLCV 1 ngay
daily = make_ohlcv("1D")

day_file = (
    DAY_DIR / "ETHUSDT_1d_2025-2026.parquet"
)

daily.to_parquet(
    day_file,
    index=False,
    compression="zstd"
)

# 5. Kiem tra
print("\n===== T06 FINAL AGGREGATION =====")
print("OHLCV 1m:", len(df))
print("OHLCV 1h:", len(hourly))
print("OHLCV 1d:", len(daily))

print("\n===== TRADE COUNT CHECK =====")
print("1m:", f"{df['trade_count'].sum():,}")
print("1h:", f"{hourly['trade_count'].sum():,}")
print("1d:", f"{daily['trade_count'].sum():,}")

assert len(hourly) == 655 * 24
assert len(daily) == 655

assert (
    df["trade_count"].sum()
    == hourly["trade_count"].sum()
    == daily["trade_count"].sum()
), "Tong giao dich khong khop!"

print("\nCHECK PASSED!")
print("File 1h:", hour_file)
print("File 1d:", day_file)

display(daily.head())


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path(
    "/content/drive/MyDrive/onchain-whale-analysis/data/market"
)

report = {
    "task": "T06",
    "status": "aggregation_completed",
    "source": "Binance Vision - Spot Trades",
    "symbol": "ETHUSDT",
    "timezone": "UTC",
    "date_start": "2025-01-01",
    "date_end": "2026-10-17",
    "days": 655,
    "raw_trades": 2435744778,
    "raw_parquet_gib": 24.33,
    "ohlcv": {
        "1m_rows": 943200,
        "1h_rows": 15720,
        "1d_rows": 655
    },
    "checks": {
        "missing_daily_files": 0,
        "metadata_errors": 0,
        "trade_count_consistent": True,
        "full_trade_level_quality_check": False
    },
    "paths": {
        "raw": "data/market/trades/",
        "1m": "data/market/processed/ohlcv_1m/",
        "1h": "data/market/processed/ohlcv_1h/",
        "1d": "data/market/processed/ohlcv_1d/"
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

output = ROOT / "logs/t06_final_summary.json"
output.parent.mkdir(parents=True, exist_ok=True)

output.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("DA LUU BAO CAO BAN GIAO T06")
print(output)
